
# ETD2MARC - Process and upload corrected invalid MARC

- once invalid MARC from RR_ETD_to_MARC.ipynb has been corrected, save the file as "corrected_MARC..." following the naming pattern for that semester
- upload to a subfolder in the semester folder called 'corrected' and then run this script

Inputs:
- name of semester subfolder
- y or n to upload to OCLC

Outputs:
- error list
- still invalid list
- still invalid XML
- valid XML

The valid records that are successfully uploaded to OCLC are appended to the processed list spreadsheet for the semester.


## Dependencies


In [ ]:
# connect to Google Drive. The XSLT stylesheet must be in your Google Drive to run this script.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from lxml import etree

!pip install bookops-worldcat
from bookops_worldcat import MetadataSession, WorldcatAccessToken

from io import BytesIO, StringIO

import requests
import yaml
import csv
import json
import pandas as pd

from openpyxl import load_workbook

from logging import exception


from google.colab import runtime

## !! ACTION REQUIRED: enter semester subfolder when prompted


In [ ]:
new_folder = input("Subfolder for this semester e.g. '2025_Summer': ")

In [ ]:
# API config file
configfile = "/content/drive/My Drive/ETD2MARC/metadata_api_config.yml"

# direct output of XSLT script - all MARC/XML records
MARCXMLfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/corrected/corrected_MARC_"+new_folder+".xml"

# valid corrected MARC/XML
validXMLfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/corrected/valid_corrected_MARC_"+new_folder+".xml"

# still invalid MARC/XML
invalidXMLfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/corrected/still_invalid_MARC_"+new_folder+".xml"

# list of invalid MARC records
invalidfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/corrected/still_invalid_list_"+new_folder+".xlsx"

# list of records that failed to upload to WMS
errorfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/corrected/error_corrected_list_"+new_folder+".xlsx"

# list of records successfully uploaded to WMS
processedfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/processed_list_"+new_folder+".xlsx"


## API call

In [ ]:
# set up lxml etrees for valid and invalid MARC
marcNSMAP = {'marc' : 'http://www.loc.gov/MARC21/slim'}
invalidRoot = etree.Element("{http://www.loc.gov/MARC21/slim}collection", nsmap=marcNSMAP)
validRoot = etree.Element("{http://www.loc.gov/MARC21/slim}collection", nsmap=marcNSMAP)
errorRoot = etree.Element("{http://www.loc.gov/MARC21/slim}collection", nsmap=marcNSMAP)

In [ ]:
# start API session
with open(configfile, 'r') as stream:
    config = yaml.safe_load(stream)

    token = WorldcatAccessToken(
        key= config.get('key'),
        secret= config.get('secret'),
        scopes="WorldCatMetadataAPI:manage_bibs",
    )
    print(token)
    print(token.is_expired())

## VALIDATE MARC RECORD

In [ ]:
# function to retrieve JSON error output from BookOps-WorldCat
# solution from https://www.reddit.com/r/learnpython/comments/s20v6w/easy_way_to_extract_json_part_of_a_string/

def parse_json(s):
    s = s[next(idx for idx, c in enumerate(s) if c in "{["):]
    try:
        return json.loads(s)
    except json.JSONDecodeError as e:
        return json.loads(s[:e.pos])

In [ ]:
invalidList = {
    'url':[],
    'error':[]
}

# validate record
with open(MARCXMLfile,"rb") as xml_file:
    session = MetadataSession(authorization=token)
    marcCollection = BytesIO(xml_file.read())
    tree = etree.parse(marcCollection)
    root = tree.getroot()
    for marcRecord in root.iterfind("{http://www.loc.gov/MARC21/slim}record"):
      # if record validates, add to the valid etree
      try:
        response = session.bib_validate(
        record = etree.tostring(marcRecord),
        recordFormat="application/marcxml+xml",
        validationLevel="validateFull",
        )
        print(response.json())
        if response.json()["status"]["summary"] == 'VALID':
          validRoot.append(marcRecord)
      # otherwise add to the invalid etree and add the id and error to the list of invalid records
      except Exception as e:
        error_json = parse_json(str(e))
        print(error_json)

        invalidRoot.append(marcRecord)
        invalidList['url'].append(marcRecord.xpath("./*[local-name() ='datafield'][@tag='856']/*[local-name() = 'subfield'][@code='u']/text()"))
        invalidList['error'].append(error_json['validationErrors']['errors'])

# save invalid list to file
df = pd.DataFrame(invalidList)
df.to_excel(invalidfile, index=False)

# save MARC/XML to files
with open(invalidXMLfile, 'wb') as fp:
  fp.write(etree.tostring(invalidRoot, pretty_print="true", encoding="utf-8"))

with open(validXMLfile, 'wb') as fp:
  fp.write(etree.tostring(validRoot, pretty_print="true", encoding="utf-8"))


## !!CAUTION - CREATE MARC RECORD

In [ ]:
check = input("Ready to upload via API? (y/n): ")

if check != 'y':
  runtime.unassign()

else:
  errorList = {
      'url':[],
      'error':[]
  }

  processedList = {
      'OCN':[],
      'DOI':[]
  }

  session = MetadataSession(authorization=token)

  for marcRecord in validRoot.iterfind("{http://www.loc.gov/MARC21/slim}record"):
    try:
      createResponse = session.bib_create(
          record = etree.tostring(marcRecord),
          recordFormat="application/marcxml+xml"
              )
      response = createResponse.content
      responseTree = etree.parse(BytesIO(response))
      responseRoot = responseTree.getroot()
      print(responseRoot.xpath("./*[local-name() ='controlfield'][@tag='001']/text()"))
      processedList['OCN'].append(responseRoot.xpath("./*[local-name() ='controlfield'][@tag='001']/text()")[0])
      processedList['DOI'].append(marcRecord.xpath("./*[local-name() ='datafield'][@tag='856']/*[local-name() = 'subfield'][@code='u']/text()")[0])

    except Exception as e:
      error_json = parse_json(str(e))
      print(error_json)

      errorRoot.append(marcRecord)
      errorList['url'].append(marcRecord.xpath("./*[local-name() ='datafield'][@tag='856']/*[local-name() = 'subfield'][@code='u']/text()")[0])
      errorList['error'].append(error_json)

  df = pd.DataFrame(errorList)
  df.to_excel(errorfile, index=False)

  # Append processed records to this semester's processed list

  wb = load_workbook(processedfile)
  start_row = wb["Sheet1"].max_row

  processed_df = pd.DataFrame(processedList)

  with pd.ExcelWriter(processedfile, mode="a", engine="openpyxl", if_sheet_exists="overlay") as writer:
      processed_df.to_excel(writer, sheet_name="Sheet1", startrow=start_row, header=False, index=False)

